In [1]:
import pandas as pd
from openai import OpenAI

pd.set_option("display.max_colwidth", None)
import time

from tqdm.notebook import tqdm

with open("./secrets.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)

df_labelled = pd.read_csv("output/random_sample-labelled.csv")
df_labelled

,tweet_id,sample,date,content,hashtags,favorites,retweets,label
0,911,SOTU (Jan 31),31/1/2018 0:35,"Lots to watch for in tonight's # SOTU address -- reactions from FLOTUS, the # metoo movement, implications for midterms, and all the Dreamers coming as guests http://ow.ly/EqSf30i5G1M",#sotu,0,0,1
1,842,SOTU (Jan 31),31/1/2018 1:03,Is 007 too toxic for the # MeToo era? https://www.theguardian.com/film/2018/jan/30/times-up-for-james-bond-is-007-too-brutish-for-the-me-too-era?CMP=Share_iOSApp_Other …,NaN,0,0,1
2,293,SOTU (Jan 31),31/1/2018 5:05,"jealous statement, you look great, but think about the # metoo movement of the past year by women that have been victims of sexual assault, harassment or inequality. Does this really say # timesup ? No babydoll. This says I never want women to move past men’s misogyny because it",#timesup,0,0,1
3,554,SOTU (Jan 31),31/1/2018 3:11,Seattleite Leah Griffin is accompanying Sen. @ PattyMurray to bring # MeToo to # SOTU.http://bit.ly/2nkIcC2,NaN,1,0,1
4,581,SOTU (Jan 31),31/1/2018 3:01,Mentoring After # MeToo https://goo.gl/f1RqWg,NaN,0,0,1
...,...,...,...,...,...,...,...,...
95,170378,General (all periods),9/5/2018 2:16,"After shocking allegations of sexual of abuse, champion of the # MeToo movement, NY AG Eric Schneiderman resigns from office. http://www.news10.com/news/local-news/four-women-accuse-new-york-attorney-general-of-abuse/1165664376 …",NaN,1,0,1
96,276339,General (all periods),12/9/2018 21:22,So any woman who supports the second amendment (myself and MANY OTHERS) deserve to be spoken to this way? # MeToo # Crazy # maga # Stopthehate,"#crazy, #maga, #stopthehate",0,0,0
97,258327,General (all periods),19/9/2018 6:30,@ SenFeinstein Dirty Dianne wat about this accusation where are the # MeToo pink nasty hat wearing females? # biased # LiberalismIsAMentalDisease,"#biased, #liberalismisamentaldisease",0,0,0
98,181804,General (all periods),29/6/2018 0:24,"Russell Simmons, an 'Extra' Host and NBC News: A Sexual Assault Accuser's Story of Pain and Frustration https://www.hollywoodreporter.com/features/russell-simmons-an-extra-host-nbc-news-a-sexual-assault-accusers-story-1123824 … by @ THR # metoo https://www.hollywoodreporter.com/features/russell-simmons-an-extra-host-nbc-news-a-sexual-assault-accusers-story-1123824 …",NaN,0,0,1


In [66]:
def classify_tweet(tweet_text: str) -> int:
    """
    Classify a tweet as authentic or inauthentic #MeToo discourse using GPT-4o.

    Parameters
    ----------
    tweet_text : str
        The text content of the tweet to classify

    Returns
    -------
    int
        1 if classified as authentic, 0 if not authentic
    """
    prompt = f"""
    Classify this #metoo tweet as "Authentic" or "Not Authentic" based on these criteria:
    
    AUTHENTIC #metoo discourse includes:
    - Personal experiences of harassment or assault (including poetry, creative writing)
    - Institutional or workplace responses to misconduct
    - Political discussions referencing #metoo (legislation, elections, speeches)
    - Meta-commentary about movement tactics or effectiveness
    - News coverage of harassment cases or accountability (including news headlines, links to articles)
    - Academic or intellectual engagement with movement themes
    - Protest or march activity where #metoo is mentioned among related causes
    - Expression of disapproval of Trump and Republican administration
    - Tweets connecting #metoo to broader social justice movements
    - Expressions of solidarity or support
    
    INAUTHENTIC tweets include:
    - Jokes using #metoo against the movement
    - Sarcasm or ironic use of #metoo against the movement
    - Co-opting #metoo to attack Democrats
    - Dismissing survivors sharing their experiences
    - Attacking accusers as liars
    
    Tweet: {tweet_text}
    
    Answer only: Authentic or Not Authentic
    """
    response = client.chat.completions.create(
        # https://platform.openai.com/docs/api-reference/chat/create
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10,
        seed=42,
    )
    answer = response.choices[0].message.content.strip()
    if "Authentic" in answer and "Not Authentic" not in answer:
        return 1
    elif "Not Authentic" in answer:
        return 0

In [67]:
# classify
gpt_labels = []

for _, row in tqdm(df_labelled.iterrows(), total=len(df_labelled)):
    label = classify_tweet(row["content"])
    gpt_labels.append(label)

    time.sleep(0.3)

df_labelled["gpt_label"] = gpt_labels
df_labelled.to_csv("output/random_sample-labelled-gpt-4o-labels.csv")

  0%|          | 0/100 [00:00<?, ?it/s]

In [68]:
# Calculate agreement
matches = (df_labelled["label"] == df_labelled["gpt_label"]).sum()
agreement_pct = (matches / 100) * 100

print(f"Agreement: {agreement_pct:.1f}%")

Agreement: 93.0%


In [3]:
disagreements = df_labelled[df_labelled["label"] != df_labelled["gpt_label"]]

# i said authentic, llm said not
me_authentic_gpt_not = disagreements[disagreements["label"] == 1].head(10)

for idx, row in me_authentic_gpt_not.iterrows():
    print(f"Tweet: {row['content']}")
    print("lp: Authentic, LLM: Not Authentic\n")

Tweet: "There are only two options: make progress or make excuses." - @ TonyRobbins # metoo # TimesUp
lp: Authentic, LLM: Not Authentic

Tweet: YESVeto proof # DACA # CHIP NOW # IgnoreTrump # BLM # Millennials # AARP # veterans # tcot # metoo # Independent # GenX # p2 # GenZ # LGBTW # womenmatterhttps://twitter.com/megwagner/status/954818888818032640 …
lp: Authentic, LLM: Not Authentic

Tweet: @ DavidAFrench my experience from this past summer tells me that if # Kavanaugh had been accused of exposing himself while riding a bike through St. Louis, that would've been all right according to the # MeToo movement. https://pjmedia.com/trending/brett-
lp: Authentic, LLM: Not Authentic

Tweet: I’m having panic attacks! # MeToo
lp: Authentic, LLM: Not Authentic

Tweet: Eric Schneiderman's # MeToo Hypocricy https://youtu.be/iAB-HsTy7v8 via @ YouTube
lp: Authentic, LLM: Not Authentic

Tweet: I would have thought that # metoo would have showed us that no group has a monopoly on hypocrital disgusti

In [5]:
# my labels
your_authentic = (df_labelled["label"] == 1).sum()
your_not = (df_labelled["label"] == 0).sum()

# GPT's labels
gpt_authentic = (df_labelled["gpt_label"] == 1).sum()
gpt_not = (df_labelled["gpt_label"] == 0).sum()

print(f"i labeled:  {your_authentic} authentic, {your_not} not")
print(f"GPT labeled:  {gpt_authentic} authentic, {gpt_not} not")

i labeled:  79 authentic, 21 not
GPT labeled:  72 authentic, 28 not


In [20]:
disagreements

,tweet_id,sample,date,content,hashtags,favorites,retweets,label,gpt_label
5,248,SOTU (Jan 31),31/1/2018 5:37,My Classic professor just made a terrible comparison with the battle of Troy and the # Metoo movement and I am very tempted to drop this class.,NaN,0,0,1,0
9,49,SOTU (Jan 31),31/1/2018 7:28,"""There are only two options: make progress or make excuses."" - @ TonyRobbins # metoo # TimesUp",#timesup,2,0,1,0
28,24063,Women's March (Jan 21),21/1/2018 10:02,YESVeto proof # DACA # CHIP NOW # IgnoreTrump # BLM # Millennials # AARP # veterans # tcot # metoo # Independent # GenX # p2 # GenZ # LGBTW # womenmatterhttps://twitter.com/megwagner/status/954818888818032640 …,"#daca, #chip, #ignoretrump, #blm, #millennials, #aarp, #veterans, #tcot, #independent, #genx, #p2, #genz, #lgbtw",1,0,1,0
37,23890,Women's March (Jan 21),21/1/2018 11:18,# WomensMarch2018 # timesup # metoo,"#womensmarch2018, #timesup",0,0,1,0
71,304202,Kavanaugh Testimony (Sep 27),27/9/2018 1:05,"@ DavidAFrench my experience from this past summer tells me that if # Kavanaugh had been accused of exposing himself while riding a bike through St. Louis, that would've been all right according to the # MeToo movement. https://pjmedia.com/trending/brett-",#kavanaugh,0,0,1,0
84,169339,General (all periods),9/5/2018 16:38,Eric Schneiderman's # MeToo Hypocricy https://youtu.be/iAB-HsTy7v8 via @ YouTube,NaN,0,0,1,0
86,49764,General (all periods),12/1/2018 4:00,I would have thought that # metoo would have showed us that no group has a monopoly on hypocrital disgusting predators.,NaN,1,0,1,0
89,208058,General (all periods),3/6/2018 21:11,"@ CathyYoung63 who are the feminists you are reading who focus mostly on ""bashing men for trivial transgressions."" I read Ijeoma Oluo, Aminatou Sow, Ruth Bader Ginsburg, Chimamanda Ngozi Adichie, Lindy West, Caitlin Flanagan, and while they spoke out in s",NaN,0,0,1,0
